# Credit Card Default Prediction

This notebook loads the dataset from the Kaggle source (`uciml/default-of-credit-card-clients-dataset`) and trains/tests a baseline model to predict whether a customer will default on the next payment cycle.

## 1) Load dataset

The dataset (`uciml/default-of-credit-card-clients-dataset`) is downloaded from Kaggle with `kagglehub`. If Kaggle is unreachable, the notebook falls back to the local Excel file at `data/dataset.xls`. Either way the result is a DataFrame named `data`.

### Import dependencies

Import the Python libraries used throughout the notebook.

In [1]:
# Import libraries and dependencies
# (install with: pip install -r requirements.txt)

from pathlib import Path

import numpy as np
import pandas as pd

### Download and inspect the dataset

Load the Kaggle dataset, using the local copy as a fallback, and inspect its contents.

In [2]:
# Load the dataset from Kaggle with kagglehub (cached locally after the first download).
# If Kaggle can't be reached, fall back to the copy committed at data/dataset.xls.
dataset_slug = "uciml/default-of-credit-card-clients-dataset"
local_path = Path("data") / "dataset.xls"


def load_from_kaggle(slug):
    # Imported here so a missing kagglehub install falls back to the local file
    # instead of breaking the whole notebook.
    import kagglehub

    folder = Path(kagglehub.dataset_download(slug))
    csv_files = sorted(folder.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV file found in {folder}")
    frame = pd.read_csv(csv_files[0])
    # The Kaggle CSV names the target "default.payment.next.month"; the rest of this
    # notebook (and the UCI workbook) uses "default payment next month".
    frame.columns = frame.columns.str.strip().str.replace(".", " ", regex=False)
    return frame, f"Kaggle ({csv_files[0].name})"


def load_from_local(path):
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path.resolve()}")
    # The first row holds the original field names; the workbook header is on row 2.
    return pd.read_excel(path, sheet_name="Data", header=1, engine="xlrd"), str(path)


try:
    data, source = load_from_kaggle(dataset_slug)
except Exception as err:
    print(f"Kaggle download failed ({type(err).__name__}: {err}); using local copy instead.")
    data, source = load_from_local(local_path)

print(f"Loaded {data.shape[0]:,} rows and {data.shape[1]:,} columns from {source}")
display(data.head())

Loaded 30,000 rows and 25 columns from Kaggle (UCI_Credit_Card.csv)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


## 2) Prepare training data

### Define feature meanings and groups

Document the dataset columns and organize them into groups for preprocessing.

In [3]:
# Feature key (UCI "default of credit card clients"). Amounts are in NT dollars;
# monthly columns run from September 2005 (suffix 0/1) back to April 2005 (suffix 6).
months = ["September", "August", "July", "June", "May", "April"]
feature_key = {
    "ID": "ID of each client",
    "LIMIT_BAL": "Amount of given credit (individual + family/supplementary credit)",
    "SEX": "Gender (1=male, 2=female)",
    "EDUCATION": "1=graduate school, 2=university, 3=high school, 4=others, 5=unknown (raw codes 0, 5 and 6 merged)",
    "MARRIAGE": "Marital status (0=unknown (undocumented code), 1=married, 2=single, 3=others)",
    "AGE": "Age in years",
}
for i, month in zip([0, 2, 3, 4, 5, 6], months):
    feature_key[f"PAY_{i}"] = f"Repayment status in {month} 2005 (-2=no consumption, -1=pay duly, 0=revolving credit, 1..9=months of delay)"
for i, month in enumerate(months, start=1):
    feature_key[f"BILL_AMT{i}"] = f"Bill statement amount in {month} 2005"
for i, month in enumerate(months, start=1):
    feature_key[f"PAY_AMT{i}"] = f"Amount of previous payment in {month} 2005"
feature_key["default payment next month"] = "Target: default payment (1=yes, 0=no)"

missing = set(feature_key) - set(data.columns)
assert not missing, f"Columns missing from dataset: {missing}"

# Feature groups used by the preprocessing pipeline.
target_column = "default payment next month"
id_column = "ID"
categorical_features = ["SEX", "EDUCATION", "MARRIAGE"]
repayment_status_features = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
bill_amount_features = [f"BILL_AMT{i}" for i in range(1, 7)]
payment_amount_features = [f"PAY_AMT{i}" for i in range(1, 7)]
numeric_features = ["LIMIT_BAL", "AGE"] + bill_amount_features + payment_amount_features

def add_domain_features(frame):
    """Add deterministic, row-level credit-risk features before splitting."""
    enriched = frame.copy()
    limit = enriched["LIMIT_BAL"].replace(0, np.nan)

    for i in range(1, 7):
        bill = enriched[f"BILL_AMT{i}"]
        payment = enriched[f"PAY_AMT{i}"]
        enriched[f"UTILIZATION{i}"] = (bill / limit).clip(-10, 10)
        enriched[f"PAYMENT_RATIO{i}"] = (payment / bill.abs().clip(lower=1)).clip(-10, 10)

    pay_columns = [f"PAY_{i}" for i in [0, 2, 3, 4, 5, 6]]
    bill_columns = [f"BILL_AMT{i}" for i in range(1, 7)]
    payment_columns = [f"PAY_AMT{i}" for i in range(1, 7)]
    enriched["DELINQUENCY_COUNT"] = enriched[pay_columns].gt(0).sum(axis=1)
    enriched["MAX_DELINQUENCY"] = enriched[pay_columns].max(axis=1).clip(lower=-2)
    enriched["MEAN_DELINQUENCY"] = enriched[pay_columns].mean(axis=1)
    enriched["MEAN_BILL"] = enriched[bill_columns].mean(axis=1)
    enriched["BILL_TREND"] = (
        enriched[[f"BILL_AMT{i}" for i in (1, 2, 3)]].mean(axis=1)
        - enriched[[f"BILL_AMT{i}" for i in (4, 5, 6)]].mean(axis=1)
    )
    enriched["MEAN_PAYMENT"] = enriched[payment_columns].mean(axis=1)
    enriched["TOTAL_PAYMENT"] = enriched[payment_columns].sum(axis=1)
    enriched["MEAN_UTILIZATION"] = enriched[[f"UTILIZATION{i}" for i in range(1, 7)]].mean(axis=1)
    return enriched

domain_features = [
    "UTILIZATION1", "UTILIZATION2", "UTILIZATION3", "UTILIZATION4", "UTILIZATION5", "UTILIZATION6",
    "PAYMENT_RATIO1", "PAYMENT_RATIO2", "PAYMENT_RATIO3", "PAYMENT_RATIO4", "PAYMENT_RATIO5", "PAYMENT_RATIO6",
    "DELINQUENCY_COUNT", "MAX_DELINQUENCY", "MEAN_DELINQUENCY",
    "MEAN_BILL", "BILL_TREND", "MEAN_PAYMENT", "TOTAL_PAYMENT", "MEAN_UTILIZATION",
]
numeric_features = numeric_features + domain_features

pd.DataFrame.from_dict(feature_key, orient="index", columns=["description"])

,description
ID,ID of each client
LIMIT_BAL,Amount of given credit (individual + family/su...
SEX,"Gender (1=male, 2=female)"
EDUCATION,"1=graduate school, 2=university, 3=high school..."
MARRIAGE,"Marital status (0=unknown (undocumented code),..."
AGE,Age in years
PAY_0,Repayment status in September 2005 (-2=no cons...
PAY_2,Repayment status in August 2005 (-2=no consump...
PAY_3,Repayment status in July 2005 (-2=no consumpti...
PAY_4,Repayment status in June 2005 (-2=no consumpti...


### Clean codes and split the data

Preserve the repayment-status codes, remove the identifier and target from the predictors, and create stratified training and test sets.

In [ ]:
from sklearn.model_selection import train_test_split

clean = data.copy()

# EDUCATION: 4 ("others") stays its own group. Code 0 is not in the data documentation, and
# 5 and 6 are both documented as "unknown", so 0, 5 and 6 are merged into one "unknown" group (5).
clean["EDUCATION"] = clean["EDUCATION"].replace({0: 5, 6: 5})
# MARRIAGE: code 0 has no documented meaning, so it is kept as its own "unknown" group.
# PAY_* also uses -2 (no consumption) and 0 (revolving credit) beyond the documented -1..9;
# they are kept as-is and ordinal-encoded in the preprocessing step below.
clean = add_domain_features(clean)

# ID is a sample identifier, not a predictor.
X = clean.drop(columns=[id_column, target_column])
Y = clean[target_column]

# Stratify so both splits keep the ~22% default rate.
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=42
)

print(f"Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows | {X.shape[1]} features")
print(f"Default rate - train: {Y_train.mean():.3f}, test: {Y_test.mean():.3f}")

Train: 24,000 rows | Test: 6,000 rows | 23 features
Default rate - train: 0.221, test: 0.221


### Build the preprocessing pipeline

Encode categorical features, preserve the ordered repayment-status scale, and standardize numeric predictors.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# Repayment status is ordinal: -2 (no consumption) < -1 (paid duly) < 0 (revolving credit)
# < 1..9 (months of delay). The explicit category order makes the encoding follow that order
# (-2 -> 0, -1 -> 1, 0 -> 2, 1 -> 3, ...); any code outside the list maps to -1.
repayment_levels = list(range(-2, 10))
repayment_encoder = Pipeline(
    steps=[
        (
            "ordinal",
            OrdinalEncoder(
                categories=[repayment_levels] * len(repayment_status_features),
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
        ("scale", StandardScaler()),
    ]
)

# SEX, EDUCATION and MARRIAGE are nominal, so they are one-hot encoded (one 0/1 column per group).
# Repayment status is ordinal-encoded, and the amounts / age are scaled.
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("repayment", repayment_encoder, repayment_status_features),
        ("numeric", StandardScaler(), numeric_features),
    ]
)

# Check the encoding on the training data.
preprocessor.fit(X_train)
encoded_names = preprocessor.get_feature_names_out()
onehot_names = [name for name in encoded_names if name.startswith("categorical__")]
print(f"{len(encoded_names)} model inputs = {len(onehot_names)} one-hot + "
      f"{len(repayment_status_features)} ordinal repayment + {len(numeric_features)} scaled numeric")
print("One-hot columns:", onehot_names)

31 model inputs = 11 one-hot + 6 ordinal repayment + 14 scaled numeric
One-hot columns: ['categorical__SEX_1', 'categorical__SEX_2', 'categorical__EDUCATION_1', 'categorical__EDUCATION_2', 'categorical__EDUCATION_3', 'categorical__EDUCATION_4', 'categorical__EDUCATION_5', 'categorical__MARRIAGE_0', 'categorical__MARRIAGE_1', 'categorical__MARRIAGE_2', 'categorical__MARRIAGE_3']


## 3) Train and evaluate baseline model

### Train and evaluate the baseline

Fit a class-balanced logistic-regression model and report classification metrics and ROC AUC.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline

# Baseline: L1 regularization reduces coefficients for redundant bill/payment features;
# class_weight offsets the ~78/22 class imbalance.
baseline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(
            class_weight="balanced",
            solver="liblinear",
            l1_ratio=1.0,
            C=0.1,
            max_iter=1000,
        )),
    ]
)
baseline.fit(X_train, Y_train)

Y_pred = baseline.predict(X_test)
Y_proba = baseline.predict_proba(X_test)[:, 1]
print(classification_report(Y_test, Y_pred, target_names=["no default", "default"]))
print(f"ROC AUC: {roc_auc_score(Y_test, Y_proba):.3f}")

              precision    recall  f1-score   support

  no default       0.87      0.69      0.77      4673
     default       0.37      0.63      0.46      1327

    accuracy                           0.68      6000
   macro avg       0.62      0.66      0.62      6000
weighted avg       0.76      0.68      0.70      6000

ROC AUC: 0.710


## 4) Compare data-preparation choices

The same baseline model (logistic regression, `class_weight="balanced"`) is evaluated with different ways of preparing the data, so the only thing that changes between rows is the preparation. Every variant is scored with stratified 10-fold cross-validation on the **training set**, with the encoders and scalers re-fit inside each fold to avoid leakage. The test set stays untouched.

| Variant | Repayment status (PAY_*) | SEX / EDUCATION / MARRIAGE | EDUCATION / MARRIAGE codes |
|---|---|---|---|
| A (current) | ordinal | one-hot | cleaned (0/5/6 merged, MARRIAGE 0 kept as unknown) |
| B | one-hot | one-hot | cleaned |
| C | ordinal | raw integer codes, scaled | cleaned |
| D | ordinal | one-hot | uncleaned (raw codes) |

Metrics are for the "default" class (precision, recall, F1), plus overall accuracy and ROC AUC.

### Compare encoding strategies

Evaluate alternative repayment-status and demographic encodings with stratified cross-validation.

In [7]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate


def make_preprocessor(repayment="ordinal", demographics="onehot"):
    """Preprocessor for one way of encoding the repayment and demographic columns."""
    transformers = []
    scaled_columns = list(numeric_features)

    if demographics == "onehot":
        transformers.append(("demographics", OneHotEncoder(handle_unknown="ignore"), categorical_features))
    else:  # keep the raw integer codes and scale them like the other numbers
        scaled_columns += categorical_features

    if repayment == "ordinal":
        transformers.append(("repayment", repayment_encoder, repayment_status_features))
    else:
        transformers.append(("repayment", OneHotEncoder(handle_unknown="ignore"), repayment_status_features))

    transformers.append(("numeric", StandardScaler(), scaled_columns))
    return ColumnTransformer(transformers=transformers)


# Same training rows, but with the original EDUCATION / MARRIAGE codes (no cleaning).
X_train_uncleaned = add_domain_features(
    data.loc[X_train.index]
).drop(columns=[id_column, target_column])

variants = {
    "A. Ordinal repayment + one-hot demographics (current)": (make_preprocessor("ordinal", "onehot"), X_train),
    "B. One-hot repayment + one-hot demographics": (make_preprocessor("onehot", "onehot"), X_train),
    "C. Ordinal repayment + raw demographic codes": (make_preprocessor("ordinal", "raw"), X_train),
    "D. Ordinal repayment + one-hot, uncleaned codes": (make_preprocessor("ordinal", "onehot"), X_train_uncleaned),
}

scoring = {"roc_auc": "roc_auc", "accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1"}
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

rows = []
for name, (preprocess, X_variant) in variants.items():
    model = Pipeline(
        steps=[
            ("preprocess", preprocess),
        ("model", LogisticRegression(
            class_weight="balanced",
            solver="liblinear",
                l1_ratio=1.0,
            C=0.1,
            max_iter=1000,
        )),
        ]
    )
    scores = cross_validate(model, X_variant, Y_train, cv=cv, scoring=scoring)
    rows.append(
        {
            "variant": name,
            "model inputs": clone(preprocess).fit_transform(X_variant).shape[1],
            "ROC AUC": scores["test_roc_auc"].mean(),
            "ROC AUC std": scores["test_roc_auc"].std(),
            "recall (default)": scores["test_recall"].mean(),
            "precision (default)": scores["test_precision"].mean(),
            "F1 (default)": scores["test_f1"].mean(),
            "accuracy": scores["test_accuracy"].mean(),
        }
    )

comparison = pd.DataFrame(rows).set_index("variant").round(4)
display(comparison.sort_values("ROC AUC", ascending=False))

,model inputs,ROC AUC,ROC AUC std,recall (default),precision (default),F1 (default),accuracy
variant,,,,,,,
B. One-hot repayment + one-hot demographics,89,0.7707,0.0088,0.5790,0.4948,0.5334,0.7760
A. Ordinal repayment + one-hot demographics (current),31,0.7282,0.0132,0.6519,0.3823,0.4819,0.6898
"D. Ordinal repayment + one-hot, uncleaned codes",33,0.7279,0.0130,0.6512,0.3817,0.4812,0.6893
C. Ordinal repayment + raw demographic codes,23,0.7269,0.0124,0.6455,0.3842,0.4816,0.6924
